# Tugas Praktikum

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.image as mpimg
import cv2

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
# 1. Klasifikasi Burnout_Risk (Dataset PBL) dengan kernel linier, polynomial, dan RBF
raw_pbl = pd.read_csv('burnout_dataset1.csv')
num_cols = [c for c in raw_pbl.select_dtypes(include=[np.number]).columns if c not in ['Person_ID', 'Burnout_Score']]
pbl = raw_pbl[num_cols + ['Burnout_Risk']].copy()
pbl[num_cols] = pbl[num_cols].fillna(pbl[num_cols].median())

# Sampel stratified 5000 baris agar perbandingan tiga kernel berjalan efisien
pbl_samp, _ = train_test_split(pbl, train_size=5000, random_state=42, stratify=pbl['Burnout_Risk'])
pbl_samp = pbl_samp.reset_index(drop=True)

print('Ukuran dataset sampel:', pbl_samp.shape)
print('Jumlah nilai kosong:', pbl_samp.isna().sum().sum())
print(pbl_samp['Burnout_Risk'].value_counts())
pbl_samp.head()

Ukuran dataset sampel: (5000, 20)
Jumlah nilai kosong: 0
Burnout_Risk
Low         2537
Moderate    1478
High         985
Name: count, dtype: int64


,Age,Monthly_Income_USD,Work_Hours_Per_Week,Job_Satisfaction,Work_Life_Balance,Sleep_Hours,Anxiety_Score,Depression_Score,Mood_Score,Emotional_Stability,Physical_Activity_Hours,Meditation_Minutes,Screen_Time_Hours,Social_Media_Hours,Gaming_Hours,Coffee_Cups_Per_Day,Life_Satisfaction,Productivity_Score,Absenteeism_Days,Burnout_Risk
0,51.0,2814.0,42,3.0,5.0,5.4,41.0,60.0,42.0,59,4.2,41.0,10.3,1.4,1.6,2,3.0,37.0,20,Moderate
1,64.0,7126.0,56,7.0,7.0,9.6,45.0,39.0,61.0,47,9.3,36.0,8.7,4.0,3.1,8,5.0,55.0,19,Low
2,41.0,1027.0,30,2.0,6.0,5.0,6.0,16.0,84.0,100,8.7,21.0,6.7,3.8,2.7,6,7.0,74.0,7,Low
3,41.0,2894.0,35,2.0,6.0,8.3,32.0,52.0,54.0,75,5.0,34.0,8.4,6.1,5.4,8,7.0,61.0,13,Low
4,55.0,9830.0,40,9.0,10.0,4.3,39.0,28.0,67.0,53,4.0,26.0,5.2,1.5,3.9,1,5.0,76.0,22,Moderate


In [3]:
X_pbl = pbl_samp.drop(columns='Burnout_Risk')
y_pbl = pbl_samp['Burnout_Risk']

hasil_pbl_list = []
models_pbl = {}
splits_pbl = {}

for rasio, test_size in [('70:30', 0.3), ('80:20', 0.2)]:
    X_train_pbl, X_test_pbl, y_train_pbl, y_test_pbl = train_test_split(
        X_pbl, y_pbl, test_size=test_size, random_state=42, stratify=y_pbl
    )
    splits_pbl[rasio] = (X_train_pbl, X_test_pbl, y_train_pbl, y_test_pbl)

    for nama_kernel, kernel in [('Linier', 'linear'), ('Polynomial', 'poly'), ('RBF', 'rbf')]:
        model_pbl = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(kernel=kernel, C=1.0, degree=3, gamma='scale'))
        ])
        model_pbl.fit(X_train_pbl, y_train_pbl)
        pred_train_pbl = model_pbl.predict(X_train_pbl)
        pred_test_pbl = model_pbl.predict(X_test_pbl)
        models_pbl[(rasio, kernel)] = model_pbl

        hasil_pbl_list.append({
            'Rasio train:test': rasio,
            'Kernel': nama_kernel,
            'Jumlah train': len(X_train_pbl),
            'Jumlah test': len(X_test_pbl),
            'Akurasi train': accuracy_score(y_train_pbl, pred_train_pbl),
            'Akurasi test': accuracy_score(y_test_pbl, pred_test_pbl)
        })

hasil_pbl = pd.DataFrame(hasil_pbl_list)
hasil_pbl

,Rasio train:test,Kernel,Jumlah train,Jumlah test,Akurasi train,Akurasi test
0,70:30,Linier,3500,1500,0.813714,0.800667
1,70:30,Polynomial,3500,1500,0.862286,0.779333
2,70:30,RBF,3500,1500,0.858571,0.793333
3,80:20,Linier,4000,1000,0.814750,0.801000
4,80:20,Polynomial,4000,1000,0.857250,0.787000
5,80:20,RBF,4000,1000,0.856750,0.797000


In [4]:
# 2. Klasifikasi siang dan malam dengan fitur histogram
base_img = Path('../Modul/images') if Path('../Modul/images').exists() else Path('images')
daftar_citra = []
for folder_split in ['training', 'test']:
    for label, nilai_label in [('day', 1), ('night', 0)]:
        folder = base_img / folder_split / label
        for file in sorted(folder.glob('*.jpg')):
            daftar_citra.append({
                'Path': str(file),
                'Label': nilai_label,
                'SHA256': hashlib.sha256(file.read_bytes()).hexdigest()
            })

data_citra = pd.DataFrame(daftar_citra)
assert not data_citra.empty, 'Dataset citra tidak ditemukan.'
assert data_citra.groupby('SHA256')['Label'].nunique().max() == 1
jumlah_duplikat = data_citra.duplicated('SHA256').sum()
data_citra = data_citra.drop_duplicates('SHA256').reset_index(drop=True)

print('Jumlah citra awal:', len(daftar_citra))
print('Jumlah salinan duplikat:', jumlah_duplikat)
print('Jumlah citra unik:', len(data_citra))
print(data_citra['Label'].map({1: 'day', 0: 'night'}).value_counts())

Jumlah citra awal: 400
Jumlah salinan duplikat: 1
Jumlah citra unik: 399
Label
night    200
day      199
Name: count, dtype: int64


In [5]:
def extract_histogram(image_path, bins=32):
    image = mpimg.imread(image_path)
    image = cv2.resize(image, (1100, 600))
    image_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    fitur = []
    for kanal, rentang in [(0, [0, 180]), (1, [0, 256]), (2, [0, 256])]:
        hist = cv2.calcHist([image_hsv], [kanal], None, [bins], rentang).ravel()
        hist = hist / hist.sum()
        fitur.extend(hist)
    return np.asarray(fitur, dtype=np.float32)

X_hist = np.vstack([extract_histogram(file) for file in data_citra['Path']])
y_hist = data_citra['Label'].to_numpy()
print('Dimensi fitur histogram:', X_hist.shape)

Dimensi fitur histogram: (399, 96)


In [6]:
indeks_train_hist, indeks_test_hist = train_test_split(
    data_citra.index.to_numpy(), test_size=0.2, random_state=42, stratify=y_hist
)
X_train_hist = X_hist[indeks_train_hist]
X_test_hist = X_hist[indeks_test_hist]
y_train_hist = y_hist[indeks_train_hist]
y_test_hist = y_hist[indeks_test_hist]

print('Rasio train:test: 80:20')
print('Jumlah train:', len(X_train_hist))
print('Jumlah test:', len(X_test_hist))

Rasio train:test: 80:20
Jumlah train: 319
Jumlah test: 80


In [7]:
model_hist_awal = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf'))
])
model_hist_awal.fit(X_train_hist, y_train_hist)

cv_hist = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_hist = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.001, 0.01, 0.1, 1]
}
grid_hist = GridSearchCV(
    model_hist_awal, param_hist, scoring='accuracy', cv=cv_hist, n_jobs=1
)
grid_hist.fit(X_train_hist, y_train_hist)
model_hist_terbaik = grid_hist.best_estimator_

print('Parameter terbaik:', grid_hist.best_params_)
print('Akurasi validasi silang terbaik:', grid_hist.best_score_)

Parameter terbaik: {'svc__C': 1, 'svc__gamma': 'scale'}
Akurasi validasi silang terbaik: 0.9905257936507936


In [8]:
hasil_tuning_hist = pd.DataFrame(grid_hist.cv_results_)
hasil_tuning_hist[['param_svc__C', 'param_svc__gamma', 'mean_test_score', 'std_test_score']].sort_values(
    'mean_test_score', ascending=False
).head(10)

,param_svc__C,param_svc__gamma,mean_test_score,std_test_score
11,10.0,0.001,0.990526,0.012674
10,10.0,scale,0.990526,0.012674
7,1.0,0.01,0.990526,0.012674
5,1.0,scale,0.990526,0.012674
16,100.0,0.001,0.990526,0.012674
17,100.0,0.01,0.990526,0.012674
15,100.0,scale,0.990526,0.012674
12,10.0,0.01,0.990526,0.012674
6,1.0,0.001,0.987450,0.006276
0,0.1,scale,0.968651,0.000198


In [9]:
hasil_hist_list = []
for nama_model, model_hist in [
    ('RBF parameter bawaan', model_hist_awal),
    ('RBF hasil tuning CV', model_hist_terbaik)
]:
    pred_train_hist = model_hist.predict(X_train_hist)
    pred_test_hist = model_hist.predict(X_test_hist)
    hasil_hist_list.append({
        'Model': nama_model,
        'Rasio train:test': '80:20',
        'Jumlah fitur': X_hist.shape[1],
        'C': model_hist.named_steps['svc'].C,
        'Gamma': model_hist.named_steps['svc'].gamma,
        'Akurasi train': accuracy_score(y_train_hist, pred_train_hist),
        'Akurasi test': accuracy_score(y_test_hist, pred_test_hist)
    })

hasil_hist = pd.DataFrame(hasil_hist_list)
hasil_hist

,Model,Rasio train:test,Jumlah fitur,C,Gamma,Akurasi train,Akurasi test
0,RBF parameter bawaan,80:20,96,1.0,scale,1.0,0.9875
1,RBF hasil tuning CV,80:20,96,1.0,scale,1.0,0.9875
